# 03 Provider Routing and Failover Policies (OpenClaw, 2026)

## What This Lesson Is
Implement policy-driven model routing and fallback behavior through OpenClaw agent execution.

## Scientific Lens
- Concept: Risk/cost-aware model policy selection under failure conditions.
- Measure: Fallback success ratio under injected primary failures.
- Validity Limit: Policy quality depends on accurate task classification metadata.


## How It Works
1. Create routing policy matrix from task class to provider chain.
2. Inject provider failure and verify deterministic fallback choice.
3. Execute live query and inspect response quality under configured route.


In [ ]:
import os
from openai import OpenAI  # OpenAI SDK used as protocol client to OpenClaw gateway

OPENCLAW_BASE_URL = os.getenv("OPENCLAW_BASE_URL", "http://127.0.0.1:18789").rstrip("/")
OPENCLAW_GATEWAY_TOKEN = os.getenv("OPENCLAW_GATEWAY_TOKEN") or os.getenv("OPENAI_API_KEY") or ""
OPENCLAW_TOKEN_SOURCE = (
    "OPENCLAW_GATEWAY_TOKEN" if os.getenv("OPENCLAW_GATEWAY_TOKEN")
    else ("OPENAI_API_KEY" if os.getenv("OPENAI_API_KEY") else "<missing>")
)
OPENCLAW_AGENT_ID = os.getenv("OPENCLAW_AGENT_ID", "main")

print("OPENCLAW_BASE_URL:", OPENCLAW_BASE_URL)
print("OPENCLAW_GATEWAY_TOKEN source:", OPENCLAW_TOKEN_SOURCE)
print("OPENCLAW_AGENT_ID:", OPENCLAW_AGENT_ID)


def build_gateway_client() -> OpenAI:
    # OpenClaw exposes an OpenAI-compatible Chat Completions endpoint at /v1/chat/completions.
    # We use the OpenAI SDK as a transport/protocol client to OpenClaw (not directly to OpenAI).
    # OpenClaw then routes to configured downstream providers/models.
    # Docs: https://docs.openclaw.ai/gateway/openai-http-api
    return OpenAI(base_url=f"{OPENCLAW_BASE_URL}/v1", api_key=OPENCLAW_GATEWAY_TOKEN or "local-dev-token")


def ask_openclaw(prompt: str, user: str = "lesson-user", temperature: float = 0.2) -> str:
    client = build_gateway_client()
    resp = client.chat.completions.create(
        model="openclaw",  # gateway-level alias/router target
        messages=[{"role": "user", "content": prompt}],
        user=user,
        temperature=temperature,
        extra_headers={"x-openclaw-agent-id": OPENCLAW_AGENT_ID},
    )
    return resp.choices[0].message.content or ""


### Why This Uses `OpenAI` Client With `model="openclaw"`
- The `OpenAI` SDK here is used as a **protocol-compatible HTTP client**.
- Requests go to the **OpenClaw gateway** (`OPENCLAW_BASE_URL/v1`), not directly to OpenAI.
- `model="openclaw"` is a **gateway alias/router target**.
- OpenClaw establishes downstream provider connections (OpenAI/Ollama/etc.) based on its own model/policy config.


## Code Walkthrough
- `Deterministic Demo` defines and validates the decision logic.
- `Live Demo` executes a real OpenClaw agent call through the OpenAI-compatible gateway API.


In [ ]:
# Deterministic Demo
policy = {
    "critical": ["openai/gpt-5.1-codex", "anthropic/claude-sonnet-4-5"],
    "low_cost": ["ollama/qwen2.5-coder:1.5b", "openai/gpt-4.1-mini"],
}
provider_up = {"openai": False, "anthropic": True, "ollama": True}

def route(task):
    for model in policy[task]:
        provider = model.split("/",1)[0]
        if provider_up.get(provider, False):
            return model
    return None

assert route("critical") == "anthropic/claude-sonnet-4-5"
assert route("low_cost") == "ollama/qwen2.5-coder:1.5b"


In [ ]:
# Live Demo
try:
    prompt = "Given a budget-constrained coding assistant, when should fallback switch to a local model?"
    print(ask_openclaw(prompt, user="policy-test"))
except Exception as exc:
    print(f"Live demo call failed: {exc}")
    print("Set OPENCLAW_GATEWAY_TOKEN in .env (or export OPENAI_API_KEY) and rerun.")


## Applied Labs
1. Encode a safety tier and force high-risk tasks away from local models.
2. Add cooldown windows to prevent provider flapping.
3. Track per-task route decision with model, reason, and timestamp.

## Validation Checklist
- Routing logic is explicit and testable.
- Fallback order is deterministic and policy-based.
- Live call demonstrates policy reasoning through OpenClaw gateway.

## Further Reading
- OpenClaw model providers: https://docs.openclaw.ai/concepts/model-providers
- OpenClaw showcase: https://docs.openclaw.ai/start/showcase
